# Online Food Delivery Customer Analysis and Prediction

## Objective
Analyze customer characteristics and build a classification model to predict the `Output` field (`Yes`/`No`).

### Workflow
1. Load and inspect data
2. Clean duplicates and leakage-prone fields
3. Perform EDA
4. Preprocess numerical/categorical variables
5. Train Logistic Regression and Random Forest
6. Evaluate Accuracy, Precision, Recall, F1 and ROC-AUC
7. Perform 5-fold cross-validation
8. Inspect feature importance
9. Demonstrate a customer prediction

**Leakage control:** `Unnamed: 13` duplicates `Output`, so it is removed. `Feedback` is excluded from prediction because it is an outcome-related field that may not be known at prediction time.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

DATA_FILE = "online food delivery dataset.csv"
raw_df = pd.read_csv(DATA_FILE)
raw_df.head()


## 1. Dataset Inspection

In [ ]:
print("Shape:", raw_df.shape)
display(raw_df.head())
print("\nData types:")
display(raw_df.dtypes)
print("\nMissing values:")
display(raw_df.isnull().sum())
print("\nExact duplicate rows:", raw_df.duplicated().sum())


## 2. Data Cleaning

In [ ]:
df = raw_df.drop_duplicates().reset_index(drop=True).copy()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

if "Unnamed: 13" in df.columns:
    df = df.drop(columns=["Unnamed: 13"])

print("Rows after duplicate removal:", len(df))
display(df.head())


## 3. Exploratory Data Analysis

In [ ]:
counts = df["Output"].value_counts().reindex(["No","Yes"], fill_value=0)
plt.figure(figsize=(7,5))
plt.bar(counts.index, counts.values)
plt.title("Overall Output Distribution")
plt.xlabel("Output"); plt.ylabel("Customer Count")
plt.show()

print("Output percentage:")
display((df["Output"].value_counts(normalize=True)*100).round(2))


In [ ]:
occupation_summary = (
    df.assign(Output_Positive=(df["Output"]=="Yes").astype(int))
      .groupby("Occupation")
      .agg(Customers=("Output_Positive","size"), Positive_Rate=("Output_Positive","mean"))
)
occupation_summary["Positive_Rate"] *= 100
display(occupation_summary.sort_values("Positive_Rate", ascending=False))

plt.figure(figsize=(8,5))
s = occupation_summary["Positive_Rate"].sort_values(ascending=False)
plt.bar(s.index, s.values)
plt.title("Positive Output Rate by Occupation")
plt.ylabel("Positive Output Rate (%)")
plt.xticks(rotation=30, ha="right")
plt.show()


In [ ]:
customer_type_summary = (
    df.assign(Output_Positive=(df["Output"]=="Yes").astype(int))
      .groupby("Customer Type")["Output_Positive"].mean().mul(100)
      .sort_values(ascending=False)
)
display(customer_type_summary.to_frame("Positive Output Rate (%)"))


In [ ]:
income_feedback = pd.crosstab(df["Monthly Income"], df["Feedback"])
display(income_feedback)

income_feedback_pct = pd.crosstab(
    df["Monthly Income"], df["Feedback"], normalize="index"
).mul(100).round(2)
display(income_feedback_pct)

plt.figure(figsize=(9,5))
bottom = np.zeros(len(income_feedback_pct))
for col in ["Negative","Positive"]:
    vals = income_feedback_pct[col].values if col in income_feedback_pct else np.zeros(len(income_feedback_pct))
    plt.bar(income_feedback_pct.index, vals, bottom=bottom, label=col)
    bottom += vals
plt.title("Feedback Composition by Monthly Income")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=30, ha="right")
plt.legend()
plt.show()


In [ ]:
display(df[["Age","Family size","latitude","longitude","Pin code"]].describe())

plt.figure(figsize=(7,5))
plt.scatter(df["Age"], df["Family size"])
plt.title("Age vs Family Size")
plt.xlabel("Age"); plt.ylabel("Family Size")
plt.show()


## 4. Prediction Setup

Target: `Output`

Excluded:
- `Unnamed: 13`: duplicate of the target.
- `Feedback`: outcome/satisfaction information that may cause leakage.

Categorical variables use One-Hot Encoding. Numerical variables use StandardScaler.


In [ ]:
model_df = df.drop(columns=["Feedback"]).copy()
X = model_df.drop(columns=["Output"])
y = model_df["Output"].map({"No":0, "Yes":1})

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Model Training and Comparison

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test,pred,zero_division=0),
        "Recall": recall_score(y_test,pred,zero_division=0),
        "F1 Score": f1_score(y_test,pred,zero_division=0),
        "ROC-AUC": roc_auc_score(y_test,proba)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df.round(3))


## 6. Best Model Evaluation

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:,1]

print("Selected model:", best_model_name)
print(classification_report(y_test, best_pred, target_names=["No","Yes"], zero_division=0))

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.colorbar()
plt.xticks([0,1], ["Predicted No","Predicted Yes"], rotation=20)
plt.yticks([0,1], ["Actual No","Actual Yes"])
for i in range(2):
    for j in range(2):
        plt.text(j,i,cm[i,j],ha="center",va="center")
plt.show()


## 7. 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring="accuracy")
print("CV scores:", np.round(cv_scores,3))
print("Mean CV accuracy:", round(cv_scores.mean(),3))
print("CV standard deviation:", round(cv_scores.std(),3))


## 8. Feature Importance

In [ ]:
rf_pipe = fitted_models["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
feature_names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()

importance = pd.Series(
    rf_model.feature_importances_, index=feature_names
).sort_values(ascending=False).head(15)

display(importance.to_frame("Importance"))

vals = importance.sort_values()
plt.figure(figsize=(9,6))
plt.barh(vals.index, vals.values)
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.show()


## 9. Example Customer Prediction

In [ ]:
sample_customer = X_test.iloc[[0]]
actual = "Yes" if y_test.iloc[0] == 1 else "No"
predicted = best_model.predict(sample_customer)[0]
probability = best_model.predict_proba(sample_customer)[0,1]

print("Actual Output:", actual)
print("Predicted Output:", "Yes" if predicted==1 else "No")
print("Probability of Yes:", round(probability,3))


## 10. Conclusion

The project provides a complete customer analytics and prediction workflow. Duplicate records are removed, target leakage is controlled, customer patterns are explored, and two classification algorithms are evaluated. The final model should be interpreted using multiple metrics and cross-validation rather than accuracy alone.

### Limitations
- Small cleaned sample size.
- Target imbalance.
- No transaction-level fields such as order value or delivery time.
- Geographic variables are observational.
- Results may change on new datasets.
